# The corpus count: which Python the dialect must admit

A read-only Python dialect admits a program by its AST node types (a whitelist, checked before anything runs) and then
gates every attribute, call and import at run time. Before writing either, this notebook counts what models actually
write: every `python_exec` cell found in the bench sweeps, the e2e artifacts, the run exports in `tmp/` and the bench
store, parsed the way the runtime parses it (as the body of a function, so a top-level `return` is legal).

The measure that matters is the fraction of cells a candidate whitelist admits WHOLE, since a cell with one excluded
node goes to a person. Every fraction is given over UNIQUE cells (the same text written twice counts once) and over
OCCURRENCES (how often a cell ran); the bench repeats the same tasks across models, so occurrences weight those tasks.

In [1]:
import json
import pandas as pd
import wmlnb
import corpus

HERE = wmlnb.ROOT / "notebooks" / "python-dialect"
DATA = HERE / "data" / "corpus-count"
OUT = HERE / "data"
pd.set_option("display.width", 160, "display.max_colwidth", 80, "display.max_rows", 80)

SOURCE = {"reader": "notebooks/python-dialect/corpus.py collect()",
          "sources": "every ~/git/window-ml* clone: tests/e2e/artifacts/**/run.json and *.md, tmp/**/*.md and *.zip; the bench store's traces/**/run.json not already found locally",
          "privacy": "features only: no cell text, no string literals"}
rows = wmlnb.extract(DATA / "cells.json", corpus.collect, source=SOURCE)
stats = wmlnb.extract(DATA / "stats.json", lambda: [corpus.last_stats()], source={"reader": "corpus.last_stats() after the cells extract"})[0]
cells = pd.DataFrame(rows)
ok = cells[cells.parse == "ok"].copy()
import numpy as _np
print(f"parsed with CPython {__import__('sys').version.split()[0]}'s ast; methods attributed with pandas {pd.__version__} and numpy {_np.__version__} "
      "(Pyodide carries CPython 3.14.0, pandas 3.0.2, numpy 2.4.3: the same names for this purpose)")
pd.DataFrame(wmlnb.provenance(DATA / "cells.json", DATA / "stats.json"))

parsed with CPython 3.13.15's ast; methods attributed with pandas 3.0.6 and numpy 2.5.3 (Pyodide carries CPython 3.14.0, pandas 3.0.2, numpy 2.4.3: the same names for this purpose)


,extract,sha256,rows,extracted_at,commit,privacy,reader,sources
0,notebooks/python-dialect/data/corpus-count/cells.json,47f6d407b058b39d,1714,2026-10-15T12:04:23+00:00,b1840bcb8f89,"features only: no cell text, no string literals",notebooks/python-dialect/corpus.py collect(),"every ~/git/window-ml* clone: tests/e2e/artifacts/**/run.json and *.md, tmp/..."
1,notebooks/python-dialect/data/corpus-count/stats.json,a15a50d523ed8b85,1,2026-10-15T12:04:23+00:00,b1840bcb8f89,NaN,corpus.last_stats() after the cells extract,NaN


## The corpus

Files read and cells found per source. A run seen in two places (a clone and the store, two clones) counts once: the
key is sweep/task/variant/repeat plus the run's session hash.

In [2]:
src = pd.DataFrame({"files_scanned": pd.Series(stats["files_scanned"])})
print(src.to_string(), "\n")
by_source = pd.DataFrame({
    "occurrences": pd.Series(stats["cells_after_dedupe"]),
    "unique_cells": pd.Series({s: int(cells.sources.map(lambda d: s in d).sum()) for s in stats["cells_after_dedupe"]}),
})
by_source.loc["all"] = [int(cells.n.sum()), len(cells)]
print(by_source.to_string(), "\n")
print(f"store error: {stats['store_error']}; malformed files: {stats['malformed_files']}")
print(f"syntax errors: {int((cells.parse != 'ok').sum())} unique cells ({int(cells[cells.parse != 'ok'].n.sum())} occurrences), left out below")
models = pd.Series([c for d in cells.models for c in d.values()], index=[m for d in cells.models for m in d], dtype=int).groupby(level=0).sum().sort_values(ascending=False)
print(f"models: {len(models)}; top 10 by occurrences:")
models.head(10)

               files_scanned
run.json                2784
run.md                   893
store_fetched             98
store_listed            2514
zip                       32 

       occurrences  unique_cells
bench         2151          1607
e2e             61            58
store          115           103
tmp             91            55
all           2418          1714 

store error: None; malformed files: 0
syntax errors: 17 unique cells (21 occurrences), left out below
models: 18; top 10 by occurrences:


glm-4.7-flash:latest                  777
openrouter.minimax/minimax-m3         430
qwen3.6:35b                           418
gemma4:31b                            140
deepseek.deepseek-flash               136
litellm.google/gemini-flash-latest     88
openrouter.openai/gpt-6-luna           72
litellm.google/gemini-3.6-flash        60
deepseek.deepseek-v4-pro               55
openrouter.z-ai/glm-5.3-flash          54
dtype: int64

## Node types

Each node type's share of cells: the fraction of unique cells containing it at least once. This is the raw material
for the whitelist; `data/node-counts.json` holds every type.

In [3]:
def share(pred, frame=ok):
    m = frame.apply(pred, axis=1)
    return m.mean(), (frame.n * m).sum() / frame.n.sum()

node_cells = pd.Series({k: 0 for d in ok.nodes for k in d})
for d in ok.nodes:
    for k in d:
        node_cells[k] += 1
node_total = pd.Series({k: 0 for k in node_cells.index})
for d in ok.nodes:
    for k, v in d.items():
        node_total[k] += v
nodes = pd.DataFrame({"cells": node_cells, "share": (node_cells / len(ok)).round(4), "total": node_total}).sort_values("cells", ascending=False)
(OUT / "node-counts.json").write_text(json.dumps({"unique_cells": len(ok), "nodes": nodes.reset_index(names="node").to_dict("records")}, indent=2) + "\n")
nodes.head(45)

,cells,share,total
Name,1685,0.9929,14090
Call,1633,0.9623,6349
Constant,1607,0.9470,11417
Attribute,1497,0.8821,4280
Assign,1434,0.8450,3057
Subscript,1418,0.8356,3045
Expr,989,0.5828,1606
alias,982,0.5787,1177
Return,918,0.5410,937
Import,860,0.5068,905


## Candidate whitelists

`STARLARK` is the halting set the spec names, as Python nodes: every statement and expression EXCEPT the ones listed in
`EXCLUDED`. Starlark proper also lacks `try`, generator expressions and `with`, so those start out excluded and the
tables below say what admitting each would buy. Recursion is not a node: the spec admits it to a bounded depth, so it
excludes nothing here.

`RULED` is the set as decided on 2026-10-15 (repl-specs; `docs/spec/PYTHON_DIALECT.md`): `STARLARK`, plus a generator
expression ONLY in consumer position, as the sole POSITIONAL argument (keyword arguments such as `key=` or
`default=` allowed: they never receive the generator) of a known consumer (`sum`, `any`, `all`, `min`, `max`,
`sorted`, `list`, `tuple`, `set`, `dict`, `enumerate`, `zip`, `str.join`), where the pre-pass rewrites it to a list
comprehension. A genexp anywhere else is refused. `while` and `class` stay refused statically: no cell in the corpus
uses either (the first line below), which confirms the refusal costs nothing, and the rungs that admitted them are
dropped.

A cell is admitted by a whitelist when it parses, every node it contains is listed, and (the `+gate` column) it would
also pass the static parts of the run-time gate: no dunder attribute or name, no call to a builtin the gate refuses
(one-argument `type(x)` is allowed, ruled the same day, and any other form is refused),
every import in `MODULES`, counting a module the prelude binds (`pd`, `np`, `io`, `sys`, …) that the cell uses without importing it. The `+gate` sets are a guess at the first allow lists, not the gate itself.

In [4]:
import ast
# Concrete node classes only: no abstract bases (stmt, expr, …), no deprecated aliases (Num, Str, …), no top-level
# containers (Module, Expression, …), which a cell's body never contains.
DEPRECATED = {"Num", "Str", "Bytes", "NameConstant", "Ellipsis", "Index", "ExtSlice", "Suite", "Param", "AugLoad", "AugStore"}
CONTAINERS = {"Module", "Interactive", "Expression", "FunctionType"}
ABSTRACT = {"AST", "mod", "stmt", "expr", "expr_context", "boolop", "operator", "unaryop", "cmpop", "excepthandler",
            "type_ignore", "type_param", "pattern", "slice"}
ALL_NODES = {n for n in dir(ast) if not n.startswith("_") and isinstance(getattr(ast, n), type)
             and issubclass(getattr(ast, n), ast.AST) and n not in ABSTRACT | DEPRECATED | CONTAINERS}
EXCLUDED = {
    "While": "may not halt; the spec refuses it statically",
    "ClassDef": "a class builds a new type with its own dunders",
    "AsyncFunctionDef": "async", "AsyncFor": "async", "AsyncWith": "async", "Await": "async",
    "Yield": "a generator suspends a frame", "YieldFrom": "a generator suspends a frame",
    "GeneratorExp": "a lazy generator object (Starlark has none)",
    "Try": "Starlark has none; the spec leaves it open", "TryStar": "except*", "ExceptHandler": "part of Try",
    "With": "a context manager runs __enter__/__exit__", "withitem": "part of With",
    "Global": "writes a module-level name", "Nonlocal": "writes an enclosing frame's name",
    "NamedExpr": "the walrus: an assignment inside an expression",
    "Match": "structural pattern matching", "TypeAlias": "a type statement", "TypeIgnore": "a type comment",
    "TypeVar": "PEP 695 type parameters", "ParamSpec": "PEP 695 type parameters", "TypeVarTuple": "PEP 695 type parameters",
}
STARLARK = {n for n in ALL_NODES if n not in EXCLUDED and not n.startswith(("match_", "Match")) and n not in {"TemplateStr", "Interpolation"}}
MODULES = {"pandas", "numpy", "math", "statistics", "re", "json", "collections", "itertools", "functools", "datetime",
           "decimal", "fractions", "string", "textwrap", "operator", "random", "bisect", "heapq", "calendar",
           "io", "base64",  # io and base64 for their in-memory parts only (StringIO, BytesIO, b64encode); a guess
           "csv", "PIL"}    # ruled in 2026-10-15, as manifest LEAVES (both are C underneath), not a bare import allowance
REFUSED_BUILTINS = {"eval", "exec", "compile", "open", "__import__", "globals", "locals", "vars", "input", "breakpoint",
                    "help", "exit", "quit", "memoryview", "setattr", "delattr"}

def top(name):
    return name.split(".")[0]

def gate_ok(r):
    return (not r.dunder_attrs and not r.dunder_names and not (set(r.builtins) & REFUSED_BUILTINS)
            and not r.type_calls["other"] and all(top(m) in MODULES for m in [*r.imports, *r.prelude]))

def admitted(wl, gate=False, consumer_genexp=False):
    """A predicate on a cell: inside `wl`, with a consumer-position genexp counted as rewritten when asked."""
    def f(r):
        ns = set(r.nodes)
        if consumer_genexp and r.genexp["other"] == 0:
            ns.discard("GeneratorExp")
        return ns <= wl and (not gate or gate_ok(r))
    return f

RULED = (STARLARK, True)
LADDER = {
    "starlark": (STARLARK, False),
    "ruled: + genexp in consumer position": RULED,
    "(any genexp, for comparison)": (STARLARK | {"GeneratorExp"}, False),
}
def row(spec):
    wl, cg = spec
    u, o = share(admitted(wl, consumer_genexp=cg)); gu, go = share(admitted(wl, gate=True, consumer_genexp=cg))
    return {"unique": round(u, 4), "occurrences": round(o, 4), "unique +gate": round(gu, 4), "occurrences +gate": round(go, 4)}
ge = pd.DataFrame(list(ok.genexp))
print(f"while: {int(ok.nodes.map(lambda d: 'While' in d).sum())} cells, class: {int(ok.nodes.map(lambda d: 'ClassDef' in d).sum())} cells")
print(f"generator expressions: {int(ge.sum().sum())} in {int((ge.sum(axis=1) > 0).sum())} cells; in consumer position {int(ge.consumer.sum() + ge.consumer_kw.sum())} "
      f"({int(ge.consumer_kw.sum())} of them beside keyword arguments, counted as consumer); elsewhere {int(ge.other.sum())}, in {int((ge.other > 0).sum())} cells")
pd.DataFrame({k: row(v) for k, v in LADDER.items()}).T

while: 0 cells, class: 0 cells
generator expressions: 177 in 172 cells; in consumer position 164 (0 of them beside keyword arguments, counted as consumer); elsewhere 13, in 13 cells


,unique,occurrences,unique +gate,occurrences +gate
starlark,0.8939,0.9128,0.8598,0.8828
ruled: + genexp in consumer position,0.9835,0.9866,0.9446,0.9516
"(any genexp, for comparison)",0.9906,0.9921,0.9511,0.9566


## What excludes the rest

For every cell `STARLARK` does not admit, the excluded nodes it contains. `only` counts cells whose ONE excluded node is
this one, so admitting it alone admits them; `gain` is the unique-cell share admitted by `STARLARK` plus this node.

In [5]:
ok["excluded"] = ok.nodes.map(lambda d: sorted(set(d) - STARLARK))
out = ok[ok.excluded.map(len) > 0]
gain = {}
for node in sorted({x for e in out.excluded for x in e}):
    has = out.excluded.map(lambda e: node in e)
    only = out.excluded.map(lambda e: e == [node])
    gain[node] = {"cells": int(has.sum()), "occurrences": int(out.n[has].sum()), "only": int(only.sum()),
                  "gain": round(share(admitted(STARLARK | {node}))[0] - share(admitted(STARLARK))[0], 4),
                  "why excluded": EXCLUDED.get(node, "")}
excl = pd.DataFrame(gain).T.sort_values("cells", ascending=False)
print(f"{len(out)} of {len(ok)} unique cells are outside STARLARK")
excl

180 of 1697 unique cells are outside STARLARK


,cells,occurrences,only,gain,why excluded
GeneratorExp,172,200,164,0.0966,a lazy generator object (Starlark has none)
With,7,8,0,0.0,a context manager runs __enter__/__exit__
withitem,7,8,0,0.0,part of With
ExceptHandler,6,7,0,0.0,part of Try
Try,6,7,0,0.0,Starlark has none; the spec leaves it open
Await,2,3,0,0.0,async
NamedExpr,1,1,0,0.0,the walrus: an assignment inside an expression


Greedy: the order in which admitting one more construct admits the most further cells. This is the ranking to read
against the spec's reasons for excluding each one; it is not a proposal to admit them in this order.

In [6]:
wl, steps = set(STARLARK), []
base = share(admitted(wl))[0]
steps.append({"add": "(starlark)", "unique": round(base, 4)})
for _ in range(6):
    cand = {n: share(admitted(wl | {n}))[0] for n in sorted({x for e in out.excluded for x in e} - wl)}
    if not cand:
        break
    n, v = max(cand.items(), key=lambda kv: kv[1])
    if v <= steps[-1]["unique"]:
        break
    wl.add(n); steps.append({"add": n, "unique": round(v, 4)})
pd.DataFrame(steps)

,add,unique
0,(starlark),0.8939
1,GeneratorExp,0.9906
2,Await,0.9918
3,NamedExpr,0.9923
4,ExceptHandler,0.9923
5,Try,0.9959


## The run-time gate's static half

What the `+gate` columns refuse, by reason, among cells `RULED` admits by node type: these are the cells the
whitelist lets through and the gate would still send to a person, so they are what the first allow lists must decide.
The last table ranks single refusers (one dunder, one builtin, one module) by the cells each sends to a person.

In [7]:
inside = ok[ok.apply(admitted(STARLARK, consumer_genexp=True), axis=1)]
reasons = {
    "dunder attribute": inside.dunder_attrs.map(bool),
    "dunder name": inside.dunder_names.map(bool),
    "refused builtin": inside.builtins.map(lambda b: bool(set(b) & REFUSED_BUILTINS)),
    "type() not one-argument": inside.type_calls.map(lambda t: t["other"] > 0),
    "import not in MODULES": inside.apply(lambda r: any(top(m) not in MODULES for m in [*r.imports, *r.prelude]), axis=1),
}
print(pd.DataFrame({k: {"cells": int(v.sum()), "share of admitted": round(v.mean(), 4)} for k, v in reasons.items()}).T.to_string(), "\n")
cnt = lambda col: pd.Series([x for xs in inside[col] for x in xs]).value_counts()
print("dunder attributes:", dict(cnt("dunder_attrs").head(10)))
print("dunder names:", dict(cnt("dunder_names").head(10)))
print("refused builtins:", dict(pd.Series([k for b in inside.builtins for k in b if k in REFUSED_BUILTINS]).value_counts()))
print("imports outside MODULES (with prelude names used):", dict(pd.Series([top(m) for _, r in inside.iterrows() for m in {*r.imports, *r.prelude} if top(m) not in MODULES]).value_counts().head(15)))
def refusers(r):
    out = {f"dunder .{a}" for a in r.dunder_attrs} | {f"dunder name {n}" for n in r.dunder_names}
    out |= {f"builtin {b}()" for b in r.builtins if b in REFUSED_BUILTINS}
    out |= {f"import {top(m)}" for m in [*r.imports, *r.prelude] if top(m) not in MODULES}
    if r.type_calls["other"]:
        out.add("type() not one-argument")
    return out
refused = pd.Series([x for _, r in inside.iterrows() for x in refusers(r)]).value_counts()
print(f"\nadmitted by RULED nodes: {len(inside)} cells; refused by the static gate: {int(inside.apply(lambda r: bool(refusers(r)), axis=1).sum())}")
refused.head(10).rename_axis("refuser").rename("cells").reset_index()

                         cells  share of admitted
dunder attribute           7.0             0.0042
dunder name                2.0             0.0012
refused builtin            5.0             0.0030
type() not one-argument    0.0             0.0000
import not in MODULES     54.0             0.0324 

dunder attributes: {'__name__': np.int64(5), '__str__': np.int64(2)}
dunder names: {'__import__': np.int64(2)}
refused builtins: {'globals': np.int64(3), '__import__': np.int64(2)}
imports outside MODULES (with prelude names used): {'scipy': np.int64(12), 'urllib': np.int64(11), 'sys': np.int64(10), 'os': np.int64(4), 'secrets': np.int64(4), 'sympy': np.int64(4), 'requests': np.int64(3), 'pyarrow': np.int64(2), 'cmath': np.int64(2), 'bs4': np.int64(1), 'subprocess': np.int64(1)}

admitted by RULED nodes: 1669 cells; refused by the static gate: 66


,refuser,cells
0,import scipy,12
1,import urllib,11
2,import sys,10
3,dunder .__name__,5
4,import os,4
5,import secrets,4
6,import sympy,4
7,import requests,3
8,builtin globals(),3
9,import pyarrow,2


## Imports and builtins

Modules by the share of cells that use them (an import, or a name the prelude binds), and builtins by the share of cells that call them.

In [8]:
imp = pd.Series([m for _, r in ok.iterrows() for m in {top(x) for x in [*r.imports, *r.prelude]}]).value_counts()
blt = pd.Series([k for b in ok.builtins for k in b]).value_counts()
pd.concat([(imp / len(ok)).round(4).head(25).rename_axis("module").rename("import share").reset_index(),
           (blt / len(ok)).round(4).head(25).rename_axis("builtin").rename("call share").reset_index()], axis=1)

,module,import share,builtin,call share
0,pandas,0.5068,print,0.4502
1,decimal,0.1019,sum,0.1344
2,io,0.0324,len,0.1120
3,numpy,0.0200,str,0.1014
4,urllib,0.0106,int,0.0819
5,csv,0.0106,float,0.0766
6,re,0.0106,round,0.0418
7,scipy,0.0071,type,0.0247
8,sys,0.0059,list,0.0230
9,PIL,0.0059,range,0.0218


## Library methods: the first manifests' name lists

Module-level names reached through an import (`pandas.DataFrame`, `numpy.where`) and methods called on values
(`.groupby`, `.str.contains`). A method name is attributed to every receiver kind whose class has it in the notebook
environment's pandas and numpy, so `.sum` lands on DataFrame, Series, GroupBy and ndarray alike: the AST cannot tell
receivers apart, and a names-only list does not need to. A name is proposed when at least `MIN_CELLS` unique cells use
it; every observed name is kept with its count.

In [9]:
import numpy as np
from pandas.core.groupby import DataFrameGroupBy, SeriesGroupBy
from pandas.core.strings.accessor import StringMethods
from pandas.core.indexes.accessors import CombinedDatetimelikeProperties
MIN_CELLS = 2

def cell_counts(col):
    c = {}
    for d in ok[col]:
        for k in d:
            c[k] = c.get(k, 0) + 1
    return pd.Series(c, dtype=int).sort_values(ascending=False)

chains, methods = cell_counts("chains"), cell_counts("methods")
public = lambda cls: {n for n in dir(cls) if not n.startswith("_")}
KINDS = {
    "pandas": {"DataFrame": public(pd.DataFrame), "Series": public(pd.Series), "Index": public(pd.Index),
               "GroupBy": public(DataFrameGroupBy) | public(SeriesGroupBy), "str": public(StringMethods),
               "dt": public(CombinedDatetimelikeProperties)},
    "numpy": {"ndarray": public(np.ndarray)},
}
BUILTIN_KINDS = {"str": public(str), "list": public(list), "dict": public(dict), "set": public(set),
                 "tuple": public(tuple), "int": public(int), "float": public(float)}

def module_names(mod):
    s = chains[[k.split(".")[0] == mod and "." in k for k in chains.index]]
    return {k[len(mod) + 1:]: int(v) for k, v in s.items()}

def kind_methods(kinds):
    out = {}
    for kind, names in kinds.items():
        acc = kind if kind in ("str", "dt") and kinds is KINDS["pandas"] else None
        got = {}
        for m, v in methods.items():
            parts = m.lstrip(".").split(".")
            if acc:
                if len(parts) == 2 and parts[0] == acc and parts[1] in names:
                    got[parts[1]] = int(v)
            elif len(parts) == 1 and parts[0] in names:
                got[parts[0]] = int(v)
        out[kind] = got
    return out

def manifest(mod):
    fns = module_names(mod)
    kinds = kind_methods(KINDS[mod])
    pick = lambda d: sorted(k for k, v in d.items() if v >= MIN_CELLS)
    return {"module": mod, "version_counted_against": (pd if mod == "pandas" else np).__version__, "min_cells": MIN_CELLS,
            "functions": pick(fns), "kinds": {k: pick(v) for k, v in kinds.items()},
            "observed": {"functions": dict(sorted(fns.items())), "kinds": {k: dict(sorted(v.items())) for k, v in kinds.items()}},
            "note": "names only; effects, return marking (fresh/alias/internal) and cost shapes are the second pass, after the CPython restricted mode"}

import decimal
import PIL.Image
KINDS["stdlib"] = {"Decimal": public(decimal.Decimal)}
KINDS["PIL"] = {"Image": public(PIL.Image.Image)}
def stdlib_manifest():
    mods = sorted(MODULES - {"pandas", "numpy"})
    fns = {m: module_names(m) for m in mods}
    kinds = kind_methods(KINDS["stdlib"])
    pick = lambda d: sorted(k for k, v in d.items() if v >= MIN_CELLS)
    return {"modules": {m: pick(f) for m, f in fns.items() if f}, "kinds": {k: pick(v) for k, v in kinds.items()},
            "leaves": {"decimal.Decimal": {"effects": "pure", "returns": "fresh", "cost": "digits",
                                           "why": "a C type used by 10% of cells; ruled a day-one entry (repl-specs 2026-10-15)"},
                       **{f"csv.{n}": {"effects": "pure", "returns": "fresh", "cost": "input length",
                                       "over": "an in-memory iterable (a StringIO or a list of lines), never a file",
                                       "why": "_csv is C; ruled in as a leaf 2026-10-15"} for n in ("reader", "writer", "DictReader")}},
            "observed": {"modules": {m: dict(sorted(f.items())) for m, f in fns.items() if f}, "kinds": kinds}, "min_cells": MIN_CELLS,
            "note": "names only, except the ruled leaf; effects, return marking and cost shapes are the second pass"}
for mod in ("pandas", "numpy"):
    m = manifest(mod)
    if mod == "pandas":
        m["dynamic"] = {"DataFrame": {"rule": "columns", "predicate": "name in obj.columns, ours, registered from the manifest, run with the depth lifted (never the object's __getattr__)",
                                      "returns": "alias(0)", "never": ["a dunder", "a name in the kind's static method set"],
                                      "why": "column attribute access (df.Region) is common; ruled 2026-10-15"}}
    (OUT / f"manifest-{mod}.json").write_text(json.dumps(m, indent=2) + "\n")
(OUT / "manifest-stdlib.json").write_text(json.dumps(stdlib_manifest(), indent=2) + "\n")
def pil_manifest():
    fns = module_names("PIL")
    # Attributed only within cells that reach PIL (an import, or the prelude's `Image`): by name alone, every
    # `str.split` in the corpus would count as `Image.split`.
    pil_cells = ok[ok.apply(lambda r: any(top(m) == "PIL" for m in [*r.imports, *r.prelude]), axis=1)]
    names = public(PIL.Image.Image)
    meths = {}
    for d in pil_cells.methods:
        for m in d:
            if m.lstrip(".") in names:
                meths[m.lstrip(".")] = meths.get(m.lstrip("."), 0) + 1
    return {"module": "PIL", "version_counted_against": PIL.__version__, "functions": sorted(fns), "kinds": {"Image": sorted(meths)},
            "leaves": {**{f"PIL.{n}": {"cost": "pixels", "returns": "fresh",
                                       **({"effects": "read(buffer arg)", "why": "open reads the path or buffer it is given: allowed only over an owned BytesIO, never a path"} if n == "Image.open" else {})}
                          for n in sorted(fns)},
                       **{f"Image.{n}": {"cost": "pixels", **({"effects": "write(buffer arg)", "why": "save writes to the file or buffer it is given: allowed only into an owned BytesIO"} if n == "save" else {})} for n in sorted(meths)}},
            "cells": len(pil_cells),
            "observed": {"functions": fns, "kinds": {"Image": meths}},
            "note": "every name the corpus calls, however rare (MIN_CELLS does not apply: the list is short); the prelude binds Image; "
                    "effects and return marking per method are the second pass, cost is in pixels (ruled 2026-10-15)"}
import PIL
(OUT / "manifest-pil.json").write_text(json.dumps(pil_manifest(), indent=2) + "\n")
print("PIL:", pil_manifest()["functions"], pil_manifest()["kinds"])
(OUT / "method-counts.json").write_text(json.dumps({"unique_cells": len(ok), "min_cells": MIN_CELLS,
    "chains": chains.to_dict(), "methods": methods.to_dict()}, indent=2) + "\n")
for mod in ("pandas", "numpy"):
    m = manifest(mod)
    print(f"{mod}: {len(m['functions'])} module names, " + ", ".join(f"{k} {len(v)}" for k, v in m["kinds"].items()))
pd.concat([chains.head(30).rename_axis("chain").rename("cells").reset_index(), methods.head(30).rename_axis("method").rename("cells").reset_index()], axis=1)

PIL: ['Image.fromarray', 'Image.open'] {'Image': ['save']}
pandas: 8 module names, DataFrame 40, Series 39, Index 23, GroupBy 12, str 9, dt 0
numpy: 7 module names, ndarray 8


,chain,cells,method,cells
0,pandas.read_csv,427,.sum,1162
1,df.head,144,.head,175
2,decimal.Decimal,143,.tolist,152
3,pandas.DataFrame,139,.astype,64
4,df.columns.tolist,126,.split,41
5,pandas.read_html,80,.to_string,37
6,df.loc,76,.str.strip,36
7,df.columns,67,.append,35
8,df.dtypes,42,.to_dict,35
9,df.shape,42,.strip,35


## The proposed whitelist

`data/whitelist.json`: the node set (`STARLARK`, the one ratchet), what stays excluded and why with the share of cells
each costs, the modules and builtins the first allow lists start from, and the methods on builtin types cells call.
Builtins are proposed when observed in at least `MIN_CELLS` cells and not in `REFUSED_BUILTINS`.

In [10]:
builtin_methods = kind_methods(BUILTIN_KINDS)
proposed = {
    "nodes": sorted(STARLARK),
    "conditional": {"GeneratorExp": "only as the sole positional argument (keywords allowed) of " + ", ".join(sorted(corpus.CONSUMERS | {"str.join"})) + "; rewritten to a ListComp by the pre-pass; refused anywhere else"},
    "type": "one-argument type(x) only: a receiver kind with no attributes, not callable, is/== and repr only; any other form refused",
    "excluded": {n: {"why": w, "cells": int(excl.loc[n, "cells"]) if n in excl.index else 0,
                     "gain_if_admitted": float(excl.loc[n, "gain"]) if n in excl.index else 0.0} for n, w in sorted(EXCLUDED.items())},
    "modules": sorted(MODULES),
    "builtins": sorted({k for k, v in blt.items() if v >= MIN_CELLS and k not in REFUSED_BUILTINS} | {"isinstance", "type"}),
    "refused_builtins": sorted(REFUSED_BUILTINS),
    "builtin_kinds": {k: sorted(n for n, v in d.items() if v >= MIN_CELLS) for k, d in builtin_methods.items()},
    "admitted": {k: row(v) for k, v in LADDER.items()},
    "decided": "2026-10-15, repl-specs rulings 1-4 (session-mail 20261015-1201Z-repl-specs-to-repl-lane3-genexp-type-columns-rulings.md)",
    "parsed_with": f"CPython {__import__('sys').version.split()[0]} ast (Pyodide carries 3.14.0; 3.14 adds only TemplateStr/Interpolation)",
}
(OUT / "whitelist.json").write_text(json.dumps(proposed, indent=2) + "\n")
{k: (len(v) if isinstance(v, (list, dict)) else v) for k, v in proposed.items()}

{'nodes': 75,
 'conditional': 1,
 'type': 'one-argument type(x) only: a receiver kind with no attributes, not callable, is/== and repr only; any other form refused',
 'excluded': 23,
 'modules': 23,
 'builtins': 20,
 'refused_builtins': 16,
 'builtin_kinds': 7,
 'admitted': 3,
 'decided': '2026-10-15, repl-specs rulings 1-4 (session-mail 20261015-1201Z-repl-specs-to-repl-lane3-genexp-type-columns-rulings.md)',
 'parsed_with': 'CPython 3.13.15 ast (Pyodide carries 3.14.0; 3.14 adds only TemplateStr/Interpolation)'}